In [1]:
##### Calculate the lat/lon coordinates of centroid of each geography

import os
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# Save path 
save_path = f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv"

In [3]:
#### Split regions into UTM's for projection based on centroid 

gdf = total_geo.copy()
gdf = gdf.to_crs("EPSG:4326")  

# Get centroid lon/lat
gdf["_cx"] = gdf.geometry.centroid.x
gdf["_cy"] = gdf.geometry.centroid.y

def utm_epsg(lon, lat): # returns the EPSG code for the appropriate UTM zone 

    zone = int((lon + 180) / 6) + 1
    if lat >= 0:
        return 32600 + zone  # WGS 84 / UTM zone N
    else:
        return 32700 + zone  # WGS 84 / UTM zone S

gdf["_epsg"] = gdf.apply(lambda r: utm_epsg(r["_cx"], r["_cy"]), axis=1)

/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_64338/2211606989.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["_cx"] = gdf.geometry.centroid.x
/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_64338/2211606989.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["_cy"] = gdf.geometry.centroid.y


In [4]:
#### Compute accurate centroids per UTM zone, then convert back to lat/lon

centroid_records = []

for epsg, group in gdf.groupby("_epsg"):
    group_proj = group.to_crs(epsg)
    centroids_proj = group_proj.geometry.centroid
    centroids_latlon = gpd.GeoSeries(centroids_proj, crs=epsg).to_crs("EPSG:4326")

    centroid_records.append(pd.DataFrame({
        "PROJ_ID": group["PROJ_ID"].values,
        "lon": centroids_latlon.x.values,
        "lat": centroids_latlon.y.values,
    }))

centroid_df = pd.concat(centroid_records, ignore_index=True)

In [5]:
### Save
centroid_df.to_csv(save_path, index=False)